# 06 — Machine Learning

## Fleet Performance Analytics Platform

This notebook applies supervised machine learning to selected fleet-management problems using the cleaned project datasets.

### Objectives

1. Predict whether a trip is likely to experience a delivery delay.
2. Estimate trip fuel consumption.
3. Estimate maintenance-event cost.
4. Classify whether a trip is profitable.

The modelling workflow emphasizes:
- reproducible train/test evaluation,
- prevention of target leakage,
- appropriate preprocessing,
- business-relevant evaluation metrics, and
- interpretation of model outputs.

> **Data note:** The project uses a synthetic dataset created for portfolio and educational purposes. Model results demonstrate the analytical workflow and should not be interpreted as production performance on a real trucking fleet.


## 1. Imports and Project Paths

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
PROJECT = Path.cwd().parent
DATA_DIR = PROJECT / "data" / "processed"
REPORT_DIR = PROJECT / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT)
print("Processed data:", DATA_DIR)


Project: /mnt/data/ml_project/fleet_performance_analysis
Processed data: /mnt/data/ml_project/fleet_performance_analysis/data/processed


## 2. Load and Validate the Data

In [2]:
trips = pd.read_csv(DATA_DIR / "trips_clean.csv")
trucks = pd.read_csv(DATA_DIR / "trucks_clean.csv")
maintenance = pd.read_csv(DATA_DIR / "maintenance_clean.csv")

print(f"Trips: {trips.shape}")
print(f"Trucks: {trucks.shape}")
print(f"Maintenance records: {maintenance.shape}")

required_trip_columns = {
    "trip_id", "truck_id", "distance_km", "cargo_weight_tons",
    "fuel_consumed_litres", "has_delay", "planned_duration_hours",
    "road_condition_score", "weather_risk_score", "route_risk_score",
    "border_crossing_flag", "profit"
}
required_truck_columns = {
    "truck_id", "age", "base_consumption_l_per_100km"
}
required_maintenance_columns = {
    "truck_id", "category", "severity", "cost", "mileage_at_maintenance"
}

assert required_trip_columns.issubset(trips.columns)
assert required_truck_columns.issubset(trucks.columns)
assert required_maintenance_columns.issubset(maintenance.columns)

print("Required columns validated successfully.")


Trips: (1500, 40)
Trucks: (35, 21)
Maintenance records: (350, 13)
Required columns validated successfully.


## 3. Modelling Principles

### Target leakage control

Features must represent information that would be available at the intended prediction point.

For example, a pre-trip delay model excludes variables such as actual delay duration, actual trip duration, fuel consumed, revenue, profit, and operating cost because these are observed during or after the trip.

Similarly, fuel efficiency is excluded from the fuel-consumption model because it is directly derived from distance and fuel consumed.

The project therefore separates **predictive inputs** from **outcomes and post-event variables**.


## 4. Prepare Trip-Level Modelling Data

In [3]:
trip_ml = trips.merge(
    trucks[["truck_id", "age", "base_consumption_l_per_100km"]],
    on="truck_id",
    how="left",
    validate="many_to_one",
)

print("Merged trip modelling dataset:", trip_ml.shape)


Merged trip modelling dataset: (1500, 42)


# Model 1 — Delivery Delay Prediction

### Business question

**Can we identify trips that are likely to experience a delivery delay using information available before or at trip planning?**

This is a binary classification problem.

**Target:** `has_delay`


In [4]:
delay_features = [
    "distance_km",
    "cargo_weight_tons",
    "planned_duration_hours",
    "age",
    "route_risk_score",
    "border_crossing_flag",
    "road_condition_score",
    "weather_risk_score",
]

X_delay = trip_ml[delay_features].copy()
y_delay = trip_ml["has_delay"].astype(int)

X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    X_delay,
    y_delay,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_delay,
)

print(f"Delay rate: {y_delay.mean():.2%}")
print(f"Training observations: {len(X_train_d)}")
print(f"Testing observations: {len(X_test_d)}")


Delay rate: 25.87%
Training observations: 1200
Testing observations: 300


## 4.1 Logistic Regression

In [5]:
logistic_delay = Pipeline([
    ("scale", StandardScaler()),
    ("model", LogisticRegression(
        class_weight="balanced",
        random_state=RANDOM_STATE,
        max_iter=2000,
    )),
])

logistic_delay.fit(X_train_d, y_train_d)
pred_logistic_delay = logistic_delay.predict(X_test_d)

delay_logistic_metrics = {
    "Model": "Logistic Regression",
    "Accuracy": accuracy_score(y_test_d, pred_logistic_delay),
    "Precision": precision_score(y_test_d, pred_logistic_delay, zero_division=0),
    "Recall": recall_score(y_test_d, pred_logistic_delay, zero_division=0),
    "F1": f1_score(y_test_d, pred_logistic_delay, zero_division=0),
}

pd.DataFrame([delay_logistic_metrics])


,Model,Accuracy,Precision,Recall,F1
0,Logistic Regression,0.746667,0.508621,0.75641,0.608247


## 4.2 Random Forest Classifier

In [6]:
rf_delay = RandomForestClassifier(
    n_estimators=400,
    random_state=RANDOM_STATE,
    class_weight="balanced",
    n_jobs=-1,
)

rf_delay.fit(X_train_d, y_train_d)
pred_rf_delay = rf_delay.predict(X_test_d)

delay_rf_metrics = {
    "Model": "Random Forest",
    "Accuracy": accuracy_score(y_test_d, pred_rf_delay),
    "Precision": precision_score(y_test_d, pred_rf_delay, zero_division=0),
    "Recall": recall_score(y_test_d, pred_rf_delay, zero_division=0),
    "F1": f1_score(y_test_d, pred_rf_delay, zero_division=0),
}

delay_results = pd.DataFrame([delay_logistic_metrics, delay_rf_metrics])
delay_results


,Model,Accuracy,Precision,Recall,F1
0,Logistic Regression,0.746667,0.508621,0.756410,0.608247
1,Random Forest,0.763333,0.559322,0.423077,0.481752


In [7]:
delay_cm = pd.DataFrame(
    confusion_matrix(y_test_d, pred_rf_delay),
    index=["Actual No Delay", "Actual Delay"],
    columns=["Predicted No Delay", "Predicted Delay"],
)

delay_cm


,Predicted No Delay,Predicted Delay
Actual No Delay,196,26
Actual Delay,45,33


### Delay Model Interpretation

For operational delay management, **recall** is particularly useful because it measures how many genuinely delayed trips are identified by the model.

The confusion matrix should be reviewed alongside precision and F1 rather than relying on accuracy alone.


In [8]:
delay_importance = (
    pd.DataFrame({
        "Feature": delay_features,
        "Importance": rf_delay.feature_importances_,
    })
    .sort_values("Importance", ascending=False)
    .reset_index(drop=True)
)

delay_importance


,Feature,Importance
0,planned_duration_hours,0.217943
1,cargo_weight_tons,0.185437
2,route_risk_score,0.152438
3,border_crossing_flag,0.108767
4,age,0.102277
5,distance_km,0.099751
6,road_condition_score,0.084902
7,weather_risk_score,0.048485


# Model 2 — Fuel Consumption Prediction

### Business question

**Can trip fuel consumption be estimated from operational information available for planning?**

This is a regression problem.

**Target:** `fuel_consumed_litres`

The model excludes direct or post-trip fuel measures, including fuel efficiency and fuel cost.


In [9]:
fuel_features = [
    "distance_km",
    "cargo_weight_tons",
    "age",
    "base_consumption_l_per_100km",
    "road_condition_score",
    "weather_risk_score",
]

X_fuel = trip_ml[fuel_features].copy()
y_fuel = trip_ml["fuel_consumed_litres"]

X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    X_fuel,
    y_fuel,
    test_size=0.20,
    random_state=RANDOM_STATE,
)

print(f"Training observations: {len(X_train_f)}")
print(f"Testing observations: {len(X_test_f)}")


Training observations: 1200
Testing observations: 300


## 5.1 Linear Regression

In [10]:
linear_fuel = Pipeline([
    ("scale", StandardScaler()),
    ("model", LinearRegression()),
])

linear_fuel.fit(X_train_f, y_train_f)
pred_linear_fuel = linear_fuel.predict(X_test_f)

fuel_linear_metrics = {
    "Model": "Linear Regression",
    "MAE": mean_absolute_error(y_test_f, pred_linear_fuel),
    "RMSE": np.sqrt(mean_squared_error(y_test_f, pred_linear_fuel)),
    "R2": r2_score(y_test_f, pred_linear_fuel),
}

pd.DataFrame([fuel_linear_metrics])


,Model,MAE,RMSE,R2
0,Linear Regression,7.869023,10.534707,0.97982


## 5.2 Random Forest Regression

In [11]:
rf_fuel = RandomForestRegressor(
    n_estimators=300,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

rf_fuel.fit(X_train_f, y_train_f)
pred_rf_fuel = rf_fuel.predict(X_test_f)

fuel_rf_metrics = {
    "Model": "Random Forest",
    "MAE": mean_absolute_error(y_test_f, pred_rf_fuel),
    "RMSE": np.sqrt(mean_squared_error(y_test_f, pred_rf_fuel)),
    "R2": r2_score(y_test_f, pred_rf_fuel),
}

fuel_results = pd.DataFrame([fuel_linear_metrics, fuel_rf_metrics])
fuel_results


,Model,MAE,RMSE,R2
0,Linear Regression,7.869023,10.534707,0.979820
1,Random Forest,6.514888,9.281582,0.984336


### Fuel Model Interpretation

For fuel planning, MAE and RMSE describe prediction error in litres, while R² summarizes the proportion of variance explained by the model.

Lower MAE/RMSE and higher R² indicate stronger predictive performance on the held-out test set.


# Model 3 — Maintenance Cost Prediction

### Business question

**Can the cost of a maintenance event be estimated from vehicle and maintenance characteristics?**

This is a regression problem.

The model uses truck age, mileage at maintenance, maintenance severity, and maintenance category.


In [12]:
maintenance_ml = maintenance.merge(
    trucks[["truck_id", "age"]],
    on="truck_id",
    how="left",
    validate="many_to_one",
)

maintenance_numeric = [
    "age",
    "mileage_at_maintenance",
    "severity",
]
maintenance_categorical = ["category"]

X_maintenance = maintenance_ml[
    maintenance_numeric + maintenance_categorical
].copy()
y_maintenance = maintenance_ml["cost"]

X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_maintenance,
    y_maintenance,
    test_size=0.20,
    random_state=RANDOM_STATE,
)

maintenance_preprocessor = ColumnTransformer([
    (
        "category",
        OneHotEncoder(handle_unknown="ignore"),
        maintenance_categorical,
    )
], remainder="passthrough")

maintenance_model = Pipeline([
    ("preprocess", maintenance_preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=400,
        random_state=RANDOM_STATE,
        min_samples_leaf=2,
        n_jobs=-1,
    )),
])

maintenance_model.fit(X_train_m, y_train_m)
pred_maintenance = maintenance_model.predict(X_test_m)

maintenance_metrics = {
    "Model": "Random Forest",
    "MAE": mean_absolute_error(y_test_m, pred_maintenance),
    "RMSE": np.sqrt(mean_squared_error(y_test_m, pred_maintenance)),
    "R2": r2_score(y_test_m, pred_maintenance),
}

pd.DataFrame([maintenance_metrics])


,Model,MAE,RMSE,R2
0,Random Forest,83.181359,130.094939,0.956514


# Model 4 — Trip Profitability Classification

### Business question

**Can a trip be classified as profitable using operational information available without directly revealing its realized profit?**

A trip is classified as:

- `1` — profitable (`profit > 0`)
- `0` — not profitable (`profit <= 0`)

The realized profit value is used only to construct the target and is excluded from the feature set.


In [13]:
trip_ml["is_profitable"] = (trip_ml["profit"] > 0).astype(int)

profit_features = [
    "distance_km",
    "cargo_weight_tons",
    "age",
    "base_consumption_l_per_100km",
    "route_risk_score",
    "border_crossing_flag",
    "road_condition_score",
    "weather_risk_score",
]

X_profit = trip_ml[profit_features].copy()
y_profit = trip_ml["is_profitable"]

X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
    X_profit,
    y_profit,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_profit,
)

rf_profit = RandomForestClassifier(
    n_estimators=400,
    random_state=RANDOM_STATE,
    class_weight="balanced",
    n_jobs=-1,
)

rf_profit.fit(X_train_p, y_train_p)
pred_profit = rf_profit.predict(X_test_p)

profit_metrics = {
    "Model": "Random Forest",
    "Accuracy": accuracy_score(y_test_p, pred_profit),
    "Precision": precision_score(y_test_p, pred_profit, zero_division=0),
    "Recall": recall_score(y_test_p, pred_profit, zero_division=0),
    "F1": f1_score(y_test_p, pred_profit, zero_division=0),
}

pd.DataFrame([profit_metrics])


,Model,Accuracy,Precision,Recall,F1
0,Random Forest,0.796667,0.833992,0.917391,0.873706


In [14]:
profit_cm = pd.DataFrame(
    confusion_matrix(y_test_p, pred_profit),
    index=["Actual Not Profitable", "Actual Profitable"],
    columns=["Predicted Not Profitable", "Predicted Profitable"],
)

profit_cm


,Predicted Not Profitable,Predicted Profitable
Actual Not Profitable,28,42
Actual Profitable,19,211


# 6. Feature Importance

Feature importance is used to identify variables that contribute most to the fitted Random Forest predictions.

These values describe **predictive importance within the model**. They do not establish causality.


In [15]:
profit_importance = (
    pd.DataFrame({
        "Feature": profit_features,
        "Importance": rf_profit.feature_importances_,
    })
    .sort_values("Importance", ascending=False)
    .reset_index(drop=True)
)

profit_importance


,Feature,Importance
0,cargo_weight_tons,0.251307
1,base_consumption_l_per_100km,0.179716
2,distance_km,0.167384
3,age,0.144870
4,route_risk_score,0.094514
5,road_condition_score,0.075986
6,weather_risk_score,0.055982
7,border_crossing_flag,0.030242


## 6.1 Maintenance Feature Importance

For the maintenance model, categorical variables are one-hot encoded before fitting. The transformed feature names are therefore extracted from the fitted preprocessing pipeline.


In [16]:
maintenance_feature_names = maintenance_model.named_steps[
    "preprocess"
].get_feature_names_out()

maintenance_importance = (
    pd.DataFrame({
        "Feature": maintenance_feature_names,
        "Importance": maintenance_model.named_steps[
            "model"
        ].feature_importances_,
    })
    .sort_values("Importance", ascending=False)
    .reset_index(drop=True)
)

maintenance_importance.head(10)


,Feature,Importance
0,category__category_Transmission Repair,0.236717
1,category__category_Oil Change,0.210790
2,category__category_Engine Service,0.146181
3,category__category_Clutch Replacement,0.090727
4,remainder__severity,0.081905
5,category__category_Tire Replacement,0.079521
6,remainder__mileage_at_maintenance,0.062668
7,category__category_Suspension Repair,0.031671
8,remainder__age,0.021050
9,category__category_Full Service,0.018107


# 7. Model Comparison Summary

In [17]:
classification_summary = pd.DataFrame([
    {"Task": "Delivery Delay", **delay_logistic_metrics},
    {"Task": "Delivery Delay", **delay_rf_metrics},
    {"Task": "Profitability", **profit_metrics},
])

regression_summary = pd.DataFrame([
    {"Task": "Fuel Consumption", **fuel_linear_metrics},
    {"Task": "Fuel Consumption", **fuel_rf_metrics},
    {"Task": "Maintenance Cost", **maintenance_metrics},
])

classification_summary


,Task,Model,Accuracy,Precision,Recall,F1
0,Delivery Delay,Logistic Regression,0.746667,0.508621,0.756410,0.608247
1,Delivery Delay,Random Forest,0.763333,0.559322,0.423077,0.481752
2,Profitability,Random Forest,0.796667,0.833992,0.917391,0.873706


In [18]:
regression_summary


,Task,Model,MAE,RMSE,R2
0,Fuel Consumption,Linear Regression,7.869023,10.534707,0.979820
1,Fuel Consumption,Random Forest,6.514888,9.281582,0.984336
2,Maintenance Cost,Random Forest,83.181359,130.094939,0.956514


# 8. Save Model Outputs

The following outputs are saved to the project `reports/` directory for reuse in reporting and portfolio documentation.


In [19]:
delay_results.to_csv(
    REPORT_DIR / "delay_model_results.csv",
    index=False,
)

delay_cm.to_csv(
    REPORT_DIR / "delay_confusion_matrix.csv",
)

delay_importance.to_csv(
    REPORT_DIR / "delay_feature_importance.csv",
    index=False,
)

fuel_results.to_csv(
    REPORT_DIR / "fuel_model_results.csv",
    index=False,
)

pd.DataFrame([maintenance_metrics]).to_csv(
    REPORT_DIR / "maintenance_model_results.csv",
    index=False,
)

maintenance_importance.to_csv(
    REPORT_DIR / "maintenance_feature_importance.csv",
    index=False,
)

pd.DataFrame([profit_metrics]).to_csv(
    REPORT_DIR / "profitability_model_results.csv",
    index=False,
)

profit_cm.to_csv(
    REPORT_DIR / "profitability_confusion_matrix.csv",
)

profit_importance.to_csv(
    REPORT_DIR / "profitability_feature_importance.csv",
    index=False,
)

print(f"Model outputs saved to: {REPORT_DIR}")


Model outputs saved to: /mnt/data/ml_project/fleet_performance_analysis/reports


# 9. Business Interpretation

The machine-learning phase provides four forms of decision support:

| Model | Operational Use |
|---|---|
| Delivery Delay Classification | Flag trips that may require additional planning attention |
| Fuel Consumption Regression | Support fuel planning and trip-level fuel estimation |
| Maintenance Cost Regression | Support maintenance budgeting and cost estimation |
| Profitability Classification | Flag trips for commercial or operational review |

Model outputs should be treated as **decision-support signals**, not automatic operational decisions.

Because the dataset is synthetic, the models require validation against real fleet data before deployment.


# 10. Conclusion

This machine-Analytical workflow demonstrates a complete applied modelling process:

```text
Business Question
 ↓
Define Prediction Target
 ↓
Select Valid Predictors
 ↓
Control Data Leakage
 ↓
Train/Test Split
 ↓
Model Training
 ↓
Prediction on Unseen Data
 ↓
Performance Evaluation
 ↓
Feature Interpretation
 ↓
Business Decision Support
```

The notebook provides reproducible classification and regression workflows that can be extended when real operational data becomes available.
